# Построение модели поведенческого скоринга для прогнозирования просрочек на 90 дней

---

**Описание исследования**

Необходимо разработать модель поведенческого скоринга для действующего портфеля потребительских кредитов. В отличие от предкредитного скоринга, данная модель должна оценивать риск возникновения дефолта (просрочки более 90 дней) на основе актуального поведения клиента. Это позволит банку заблаговременно идентифицировать проблемных заёмщиков, оптимизировать объем резервов (ожидаемый прирост ликвидности 5–10%) и снизить регуляторные риски со стороны ЦБ. Задача классификации - предсказать вероятность дефолта на горизонте 12 месяцев.

---

**Цель исследования**

Разработать и обучить модель бинарной классификации для прогнозирования возникновения просрочки платежа длительностью 90+ дней в течение следующих 12 месяцев.

- Тип задачи: Бинарная классификация.

- Целевая переменная: Факт возникновения дефолта (1 - просрочка ≥90 дней в горизонте 12 месяцев, 0 - отсутствие такой просрочки).

- Особенности таргета: Используется метод скользящего окна с горизонтом H=12 месяцев.

- Ключевой вызов: Поиск оптимального порога классификации для баланса между объемом бизнеса и уровнем риска.

- Бизнес-метрики: 
    
    - Approval Rate: Доля клиентов, признанных моделью надежными.

    - Default Rate: Доля реальных дефолтов среди одобренных клиентов.

    - Missed Defaults Rate (1−Recall): Доля пропущенных дефолтов от общего числа плохих заемщиков.

---

**Задачи исследования**

- Анализ и подготовка данных: Изучение временной оси, обработка пропусков и корректное формирование целевой переменной по методу скользящего окна.

- EDA: Исследование признаков, описывающих финансовое поведение клиентов, и поиск ранних сигналов ухудшения платежной дисциплины.

- Построение моделей: Обучение ансамблей или градиентного бустинга для детекции скрытых паттернов дефолта.

- Оптимизация порога: Настройка порога вероятности для достижения целевых показателей по Approval Rate, Default Rate и Missed Defaults Rate.

- Валидация: Проверка устойчивости модели на тестовом временном отрезке и интерпретация наиболее значимых факторов риска для бизнеса.

---

**Описание данных**

Для построения модели поведенческого скоринга используется массив из 8 таблиц. Данные включают как статичную информацию на момент выдачи кредита, так и динамические показатели, меняющиеся во времени.

1. Основа для формирования выборки и таргета

    В этих таблицах заложена логика обучения: кого предсказываем и что именно.

    - Таблица 1 (ds_15_loan_payment_credit.csv): История просрочек. Содержит ID, дату начала и длительность просрочки в днях. Используется для разметки целевой переменной (был ли факт просрочки ≥90 дней в течение года после score_date).

    - Таблица 8 (ds_15_cohort_grid.csv): Реестр дат проведения скоринга. Содержит пары ID клиента и score_date. Это «точки отсчета», для которых нужно собрать признаки и предсказать риск на 12 месяцев вперед.

2. Динамические характеристики (Поведение)

    Данные, которые обновляются ежемесячно и отражают текущее состояние дел клиента.

    - Таблица 2 (ds_15_transactions.csv): Транзакционная активность. Помесячные траты по категориям:

        - MCC_5411, 5300, 5722: Продукты, маркетплейсы, техника.

        - MCC_5814, 5812, 3990: Фастфуд, рестораны, сервисы Яндекса.

        - MCC_4900: Коммунальные платежи (ЖКУ).

    - Таблица 6 (ds_15_credit_rating.csv): Внешний кредитный рейтинг клиента на конкретные даты. Позволяет отследить, как менялась оценка надежности заемщика со стороны.

3. Социально-демографический и кредитный профиль

    Статичные или редко меняющиеся данные о заемщике.

    - Таблица 3 (ds_15_client_description.csv): Личные данные клиента (возраст, семейное положение, наличие иждивенцев) и дата, когда клиент начал пользоваться услугами банка.

    - Таблица 4 (ds_15_credit_description.csv): Финансовые условия на старте: заявленный доход и общая сумма кредита.

    - Таблица 5 (ds_15_mortgage_presence.csv): Сведения об ипотеке (наличие флага и дата открытия). Ипотека - это долгосрочный якорь, сильно влияющий на платежную дисциплину.

4. Внешние факторы

    - Таблица 7 (ds_15_macro_data.csv): Макроэкономические показатели (инфляция, безработица, ключевая ставка). Помогают модели понять, вызваны ли просрочки личными проблемами клиента или общим кризисом в стране.

---

## Перевод бизнес-задачи на язык машинного обучения

Методы и модели

- Модели: В качестве Baseline используем логистическую регрессию. Основной стек: деревья решений и бэггинг (RandomForest).

- Работа с данными: Агрегация транзакций по месяцам, создание временных лагов для кредитного рейтинга и макропоказателей.

- Threshold Tuning: Ключевой этап решения - подбор порога классификации для баланса метрик.

План решения

- Формирование витрины: Сборка всех 8 таблиц в единый датасет на основе сетки дат скоринга.

- Разметка: Расчет таргета методом скользящего окна.

- Обучение и валидация: Кросс-валидация и подбор гиперпараметров (Optuna).

- Бизнес-оптимизация: Выбор порога отсечения, удовлетворяющего требованиям банка по уровню риска.


## Загрузка необходимых библиотек

In [1]:
from pathlib import Path

In [ ]:
req_path = Path("../../requirements.txt")
if req_path.exists():
    %pip install -qr {req_path}
else:
    %pip install -qU --force-reinstall "numpy<2.2.0" "pillow<12.0" pandas==2.2.2 numba scipy scikit-learn imbalanced-learn matplotlib seaborn phik joblib category_encoders optuna mlxtend

In [2]:
import joblib
import sys

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from phik import phik_matrix

import optuna

from mlxtend.evaluate.time_series import GroupTimeSeriesSplit

import sklearn
from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.feature_selection import SelectKBest, f_regression, VarianceThreshold, RFE
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, roc_curve, roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve, CalibrationDisplay, CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler, TomekLinks

from IPython.display import display, Markdown

pd.set_option('display.max_columns', None)

In [3]:
RANDOM_STATE = 42

## Загрузка данных

In [4]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive монтирован")
except ImportError:
    print("Локальное использование")

In [5]:
def get_df(file_name):
    local_path = Path(f'/content/drive/MyDrive/datasets/{file_name}')
    yp_path = Path(f'/datasets/{file_name}')

    if local_path.exists():
        df = pd.read_csv(local_path, sep=',', decimal='.')
    elif yp_path.exists():
        df = pd.read_csv(yp_path, sep=',', decimal='.')
    else:
        raise "Путь до датасета неверный"

    int_cols = df.select_dtypes(include=['integer']).columns
    for col in int_cols:
        df[col] = pd.to_numeric(df[col], downcast='integer')

    float_cols = df.select_dtypes(include=['float']).columns
    for col in float_cols:
        df[col] = pd.to_numeric(df[col], downcast='float')

    print('-' * 70)
    print(file_name)
    display(df.head())
    df.info()

    return df

loan_payment_credit = get_df('ds_15_loan_payment_credit.csv')
transactions = get_df('ds_15_transactions.csv')
client_description = get_df('ds_15_client_description.csv')
credit_description = get_df('ds_15_credit_description.csv')
mortgage_presence = get_df('ds_15_mortgage_presence.csv')
credit_rating = get_df('ds_15_credit_rating.csv')
macro_data = get_df('ds_15_macro_data.csv')
cohort_grid = get_df('ds_15_cohort_grid.csv')

Предоставленный массив данных содержит 8 таблиц, охватывающих 13 500 клиентов и более 577 тысяч ежемесячных записей об их активности. Структура данных позволяет объединить статичные профили заемщиков (возраст, доход, семейное положение) с динамическими факторами: транзакциями по 8 категориям MCC-кодов, изменениями кредитного рейтинга и макроэкономическим контекстом. Объем целевых событий (5 500 записей о просрочках) достаточен для обучения моделей, а наличие ипотечных данных у половины базы и временной сетки скоринга дает возможность построить качественную поведенческую модель с использованием временных лагов.

## Исследовательский анализ данных

In [6]:
def show_features(df: pd.DataFrame):
    table = []
    
    num_cols, cat_cols = [], []
    for col in df.columns:
        nunique = df[col].nunique()
        if nunique > 10:
            num_cols.append(col)
        else:
            cat_cols.append(col)
        
        table.append({
            'Признак': col,
            'Уникальные значения': df[col].unique(),
            'Кол-во': nunique,
        })

    display(pd.DataFrame(table).sort_values('Кол-во', ascending=False))
    
    return num_cols, cat_cols

In [7]:
def create_num_plot(data, col):
    _, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 4))
    
    hist = axes[0]
    data.plot(
        kind='hist',
        bins=70,
        grid=True,
        ax=hist
    )
    hist.set_title(f'Распределение: {col}', fontsize=12)
    hist.set_xlabel('Значение')
    hist.set_ylabel('Количество (log)')
    hist.set_yscale('log')
    
    box = axes[1]
    sns.boxplot(
        data=data,
        ax=box
    )
    box.set_title(f'Размах: {col}', fontsize=12)
    box.set_ylabel('Значение')
    box.grid(True)
    box.tick_params(axis='x', rotation=30)

    plt.tight_layout()
    plt.show()
    plt.close()

def show_num_plots(df, cols):
    for col in cols:
        create_num_plot(df[col], col)

In [8]:
def create_cat_plot(df, col):
    plt.figure(figsize=(12, 4)) 
    
    ax = sns.countplot(
        x=col,
        data=df,
        hue=col, 
        legend=False,
        palette='viridis', 
        dodge=False,
    )
    
    plt.title(f'Распределение: {col}', fontsize=12)
    plt.xlabel('Значение')
    plt.ylabel('Количество')

    for container in ax.containers:
        ax.bar_label(container, label_type='edge', padding=1)

    plt.tight_layout()
    plt.show()
    plt.close()
    
def show_cat_plots(df, cols):
    for col in cols:
        create_cat_plot(df, col)

In [9]:
def show_missing_stats(df):
    missing_stats = pd.DataFrame({
        'Кол-во пропусков': df.isna().sum(),
        'Доля пропусков': df.isna().mean() * 100
    })
    missing_stats = missing_stats[missing_stats['Кол-во пропусков'] > 0]
    
    if missing_stats.empty:
        return "Пропусков в данных нет"
    
    return missing_stats \
            .sort_values(by='Кол-во пропусков', ascending=False) \
            .style.format({'Доля пропусков': '{:.2f}%'}) \
            .background_gradient(cmap='coolwarm')

In [10]:
def primary_analysis(df: pd.DataFrame, name):
    display(Markdown(f'### Анализ датафрейма: **{name}**'))
    
    display(Markdown(f'#### пропуски'))
    display(show_missing_stats(df))
    
    display(Markdown(f'#### дубликаты'))
    print("Явные дубликаты строк:", df.duplicated().sum())

    display(Markdown(f'#### числовые признаки'))
    num_df = df.select_dtypes(include=['number'])
    if not num_df.empty:
        display(num_df.describe())
        num_cols, cat_cols = show_features(num_df)
        
        if len(num_cols) > 0:
            show_num_plots(num_df, num_cols)

        if len(cat_cols) > 0:
            show_cat_plots(num_df, cat_cols)
    else:
        print('Отсутствуют')
            

    print()

    display(Markdown(f'#### категориальные признаки'))
    cat_df = df.select_dtypes(include=['object'])
    if not cat_df.empty:
        display(cat_df.describe())
        _, cat_cols = show_features(cat_df)

        if len(cat_cols) > 0:
            show_cat_plots(cat_df, cat_cols)
    else:
        print('Отсутствуют')
        
    print('-' * 70)

In [11]:
tables = [
    (loan_payment_credit, 'кредитный платеж'),
    (transactions, 'транзакции'),
    (client_description, 'описание клиента'),
    (credit_description, 'описание кредита'),
    (mortgage_presence, 'наличие ипотеки'),
    (credit_rating, 'кредитный рейтинг'),
    (macro_data, 'макроданные'),
    (cohort_grid, 'когортная сетка')
]

for table, name in tables:
    primary_analysis(table, name)

Вывод: технические пропуски и явные дубликаты отсутствуют во всех таблицах. Выборка включает 13 500 уникальных клиентов с равномерным распределением по возрасту (от 18 до 69 лет) и семейному статусу, при этом около 49% из них (6609 человек) имеют ипотеку. Когортная сетка подтверждает полную структуру временных рядов: для каждого из 13 500 id представлено ровно 84 ежемесячных наблюдения, что формирует массив из 577 494 записей для анализа динамики транзакций и рейтинга. В транзакционных данных по всем MCC-кодам выявлены экстремальные правосторонние выбросы (максимумы до 101 тыс. при средних значениях 6–9 тыс.), в то время как кредитный рейтинг (от 343 до 900) распределен нормально. Целевые события дефолта (5500 записей) зафиксированы строго в интервале просрочки от 80 до 150 дней, а макроэкономический фон за 7 лет отражает существенные колебания учетной ставки от 5,5% до 17%, и периоды инфляции до 4%, и дефляции (-0.54%), колебания безработицы в пределах 4,3–6,0%, так что модель сможет учесть влияние кризисов на платежеспособность.

## Объединение таблиц & Создание новых признаков

Получение информации по одному пользователю для сверки итоговой таблицы

In [12]:
id = 'IDF54995533'

display(loan_payment_credit[loan_payment_credit['ID'] == id])
display(transactions[transactions['ID'] == id])
display(client_description[client_description['ID'] == id])
display(credit_description[credit_description['ID'] == id])
display(mortgage_presence[mortgage_presence['ID'] == id])
display(credit_rating[credit_rating['ID'] == id].shift(1))
display(cohort_grid[cohort_grid['ID'] == id])

Формирование итоговой таблицы

In [164]:
def merge_tables(
    cohort_grid, 
    loan_payment_credit, 
    client_description, 
    credit_description,
    mortgage_presence, 
    macro_data,
    credit_rating,
    transactions
):
    initial_len = len(cohort_grid)
    
    for df, col in [
        (cohort_grid, 'score_date'),
        (loan_payment_credit, 'дата_начала_периода'),
        (client_description, 'дата_регистрации'),
        (mortgage_presence, 'дата_открытия'),
        (macro_data, 'date'),
        (credit_rating, 'date'),
        (transactions, 'date'),
    ]:
        df[col] = pd.to_datetime(df[col])

    first_defaults = (
        loan_payment_credit[loan_payment_credit['просрочка_дней'] >= 90]
        .sort_values(["ID", "дата_начала_периода"])
        .drop_duplicates("ID", keep="first")
        [['ID', 'дата_начала_периода']]
    )

    df = pd.merge(cohort_grid, first_defaults, on='ID', how='left')

    df['target'] = (
        (df['дата_начала_периода'] >= df['score_date']) & 
        (df['дата_начала_периода'] < df['score_date'] + pd.Timedelta(days=365))
    ).astype(int)

    df = df[~(df['дата_начала_периода'] < df['score_date'])].copy()
    after_filter_len = len(df)

    for tbl in [client_description, credit_description, mortgage_presence]:
        df = df.merge(tbl, on='ID', how='left')

    df['client_tenure_days'] = (df['score_date'] - df['дата_регистрации']).dt.days.fillna(-1)

    df['mortgage_age_days'] = (df['score_date'] - df['дата_открытия']).dt.days
    no_mortgage_mask = (df['дата_открытия'].isna()) | (df['дата_открытия'] >= df['score_date'])
    df.loc[no_mortgage_mask, 'mortgage_age_days'] = -1
    df['наличие_ипотеки'] = (~no_mortgage_mask).astype(int)

    df = df.sort_values('score_date')

    df = pd.merge_asof(
        df, macro_data.sort_values('date'),
        left_on='score_date', right_on='date',
        direction='backward', allow_exact_matches=False
    ).drop(columns=['date'])

    for tbl in [credit_rating, transactions]:
        df = pd.merge_asof(
            df, tbl.sort_values('date'),
            left_on='score_date', right_on='date', by='ID',
            direction='backward', allow_exact_matches=False
    ).drop(columns=['date'])
    df['кредитный_рейтинг'] = df['кредитный_рейтинг'].fillna(-1)

    mcc_cols = [col for col in df.columns if 'MCC_' in col]
    df[mcc_cols] = df[mcc_cols].fillna(0)
    df['total_mcc_spent'] = df[mcc_cols].sum(axis=1)
    
    df['leisure_to_base_ratio'] = (df['MCC_5814'] + df['MCC_5812']) / (df['MCC_5411'] + 1e-9)

    df['loan_to_income_ratio'] = df['сумма_кредита'] / (df['доход'] + 1e-9)

    df['spend_drop_ratio'] = df['total_mcc_spent'] / (df.groupby('ID')['total_mcc_spent'].cummax() + 1e-9)

    df = df.drop(columns=['дата_начала_периода', 'дата_открытия', 'дата_регистрации'])
    df = df.dropna()

    final_len = len(df)
    print(f"Начальный размер: {initial_len}")
    final_without_default = initial_len - after_filter_len + final_len
    print(f'После соединения всех таблиц без фильтра дефолтов: {final_without_default} (разница с исходным {initial_len - final_without_default} ({(1 - final_without_default / initial_len) * 100:.2f}%))')
    print(f"После фильтра дефолтов: {after_filter_len}")
    print(f"Итоговый размер: {final_len}")
    print(f"Потеряно строк: {initial_len - final_len} ({(1 - final_len / initial_len) * 100:.2f}%)")

    return df

In [165]:
df = merge_tables(
    cohort_grid, 
    loan_payment_credit, 
    client_description, 
    credit_description,
    mortgage_presence,
    macro_data,
    credit_rating,
    transactions
)

In [166]:
show_missing_stats(df)

In [167]:
df.head()

In [168]:
df[df['ID'] == 'IDF54995533']

Итоговая витрина сформирована с минимальной технической потерей данных: на этапе объединения таблиц отсеяно всего 152 строки (0.03%). Сокращение выборки с 577 494 до 438 227 строк (общая потеря 24.12%) является обоснованным, так как вызвано исключением клиентов, уже находившихся в дефолте на момент скоринга. Логика обработки пропусков в таблице построена на принципе исключения утечек из будущего и сохранении физического смысла данных: транзакции заполнены нулями, так как отсутствие записи означает отсутствие трат; стаж клиента, возраст ипотеки и кредитный рейтинг заполнены значением -1, чтобы модель могла четко отличить отсутствие истории или продукта от новых записей с нулевым стажем. Пропуски в макроданных (0.03%), возникшие из-за временного смещения, были удалены через dropna(), так как любой метод их заполнения (например, значениями из следующего месяца) привел бы к использованию будущей информации при прогнозировании. В результате получена чистая, обогащенная признаками таблица, полностью готовая к обучению.

## Анализ итоговой таблицы

In [169]:
eda_df = df.copy()

eda_df = eda_df.drop(columns='ID')

In [170]:
target_desc = eda_df['target'].describe()

target_desc

In [171]:
create_cat_plot(eda_df, 'target')

In [172]:
target_counts = eda_df['target'].value_counts()

print(f'Дефолта не было: {target_counts[0] / target_desc['count'] * 100:.0f}%')
print(f'Дефолт был: {target_counts[1] / target_desc['count'] * 100:.0f}%')

Явный дисбаланс классов в сторону клиентов банка, у которых не было дефолта. Соотношение почти 1:10.

In [173]:
primary_analysis(eda_df, 'Итоговый dataframe')

Анализ корреляций

In [174]:
def show_heatmap(corr, title, x, y_coef):
    num_rows = len(corr)
    dynamic_height = max(5, num_rows * y_coef)
    plt.figure(figsize=(x, dynamic_height))
    
    sns.heatmap(
        corr,
        annot=True,
        fmt='.2f',
        cmap='coolwarm',
        linewidths=.5,
        cbar=False
    )
    
    plt.title(title)
    plt.show()

In [ ]:
corr_matrix = eda_df.phik_matrix(interval_cols=['target', 'возраст', 'наличие_иждивенцев', 'доход', 'сумма_кредита', 'наличие_ипотеки', 'client_tenure_days', 'mortgage_age_days', 'учетная_ставка', 'уровень_безработицы', 'инфляция', 'кредитный_рейтинг', 'MCC_5300', 'MCC_5814', 'MCC_5812', 'MCC_5411', 'MCC_3990', 'MCC_5722', 'MCC_4900', 'MCC_другое', 'total_mcc_spent', 'leisure_to_base_ratio', 'loan_to_income_ratio', 'spend_drop_ratio'])
target_corr = corr_matrix[corr_matrix.index != 'target'][['target']].sort_values(by='target', ascending=False)

show_heatmap(target_corr, title='Корреляция с целевой переменной', x=3, y_coef=.3)

In [176]:
show_heatmap(corr_matrix, title='Матрица корреляций', x=12, y_coef=.4)

Корреляции score_date и макропоказателей не будет при обучении, так как признак score_date нужен только для разделения на фолды, а после будет удален.

client_tenure_days и mortgage_age_days + наличие_ипотеки логически связаны, но несут разный смысл. Высокая корреляция (0.97) здесь объяснима - ипотека физически не может быть старше самого клиента в базе банка. Для моделей с градиентным расчетом ошибки стоит оставить то, что коррелирует меньше, но несет смысл разных признаков: client_tenure_days и наличие_ипотеки.

Показатели MCC_ имеют также высокую корреляцию, означающую, что более обеспеченные клиенты тратят во всех сегментах больше, чем остальные и наоборот. Значит, можно создать общий один признак, обозначающий траты до score_date.

In [177]:
def drop_problematic_columns(df):
    mcc_cols = [col for col in df.columns if 'MCC_' in col]
    cols_to_drop = ['инфляция', 'учетная_ставка', 'mortgage_age_days', *mcc_cols]

    return cols_to_drop

cols_to_drop = drop_problematic_columns(eda_df)
eda_df_no_multi = eda_df.drop(columns=cols_to_drop)

In [178]:
corr_matrix = eda_df_no_multi.phik_matrix(interval_cols=['target', 'возраст', 'наличие_иждивенцев', 'доход', 'сумма_кредита', 'наличие_ипотеки', 'client_tenure_days', 'mortgage_age_days', 'уровень_безработицы', 'кредитный_рейтинг', 'total_mcc_spent', 'leisure_to_base_ratio', 'loan_to_income_ratio', 'spend_drop_ratio'])
target_corr = corr_matrix[corr_matrix.index != 'target'][['target']].sort_values(by='target', ascending=False)

show_heatmap(target_corr, title='Корреляция с целевой переменной', x=3, y_coef=.1)

In [179]:
show_heatmap(corr_matrix, title='Матрица корреляций', x=12, y_coef=.3)

TODO: Вывод по EDA

## Моделирование

### Базовые модели

1. Подготовьте обучающую, калибровочную и тестовую выборки. При разделении данных на выборки, на которых будет производиться оценка качества и калибровка, используйте размер, равный 1200.

2. При необходимости проведите категоризацию данных, применив нужный Encoder и использовав пайплайн.

3. Обучите базовые модели с кросс-валидацией по трём фолдам:
    * Две базовые модели - логистическую регрессию и случайный лес - без балансировки классов в целевой переменной.
    * Логистическую регрессию и случайный лес с балансировкой классов. Выберите метод балансировки самостоятельно. Обязательно примените хотя бы один метод. Можно попробовать несколько и выбрать лучший.
    * Сделайте выводы о работе всех четырёх моделей.

4. Случайный лес с настройками по умолчанию легко переобучается, потому что запоминает обучающую выборку, из-за чего модель может терять в качестве на новых данных. Логистическая регрессия же сразу готова к работе за счёт встроенной L2-регуляризации, которая автоматически контролирует сложность модели.

   Чтобы исправить проблемы модели Random Forest, вам нужно подобрать для неё гиперпараметры с помощью  Optuna. Количество гиперпараметров должно быть не менее трёх. Для оптимизации используйте метрику missed defaults rate.

5. Сравните все полученные модели.

6. Для оценки моделей используйте метрики:
   * accuracy или ROC-AUC,
   * approval rate,
   * default rate,
   * missed defaults rate.

7. Сделайте вывод о работе, проделанной в этом разделе.

### КОММЕНТАРИЙ К РАБОТЕ
#### Не до конца понял мысль по 1200 строк для теста и валидации, почему выбрано именно такое число, учитывая кол-во данных в датасете? Буду рад пояснению.

В задаче сказано: "Вам нужно придерживаться описанной выше логики, используя горизонт прогнозирования 12 месяцев.". В материалах курса также об этом писалось: "Длина тестовой выборки должна быть как минимум равна горизонту прогнозирования — сколько шагов вперёд вы хотите предсказывать".

In [ ]:
df = df.sort_values('score_date').reset_index(drop=True)

max_date = df['score_date'].max()
test_threshold = max_date - pd.DateOffset(years=1)
calib_threshold = test_threshold - pd.DateOffset(years=1)

df_test = df[df['score_date'] >= test_threshold].copy()
df_calib = df[(df['score_date'] >= calib_threshold) & (df['score_date'] < test_threshold)].copy()
df_train = df[df['score_date'] < calib_threshold].copy()

print(f"Обучение: {df_train['score_date'].min().date()} - {df_train['score_date'].max().date()} | Строк: {len(df_train)}")
print(f"Калиброка: {df_calib['score_date'].min().date()} - {df_calib['score_date'].max().date()} | Строк: {len(df_calib)}")
print(f"Тест: {df_test['score_date'].min().date()} - {df_test['score_date'].max().date()} | Строк: {len(df_test)}")

gtss = GroupTimeSeriesSplit(n_splits=3, test_size=12)

groups = df_train['score_date']
X = df_train.drop(columns=['target'])
y = df_train['target']
print(f"Размер всего обучающего датасета: X={X.shape}; y={len(y)}")

In [187]:
cols_to_drop = drop_problematic_columns(X) + ['ID', 'score_date']

num_cols = [col for col in X.select_dtypes(include=['number']).columns if col not in cols_to_drop]
cat_cols = [col for col in X.select_dtypes(include=['object']).columns if col not in cols_to_drop]

In [204]:
linear_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols),
    ],
    remainder='drop'
).set_output(transform="pandas")

tree_preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols)
    ],
    remainder='passthrough'
).set_output(transform="pandas")

In [202]:
X_transformed_df = linear_preprocessor.fit_transform(X)
X_transformed_df.head()

In [205]:
X_transformed_df = tree_preprocessor.fit_transform(X)
X_transformed_df.head()

In [206]:
def get_approval_rate(fn, tn, n):
    return (fn + tn) / n

def get_default_rate(fn, tn):
    return fn / (fn + tn)

def get_missed_defaults_rate(fn, tp):
    return fn / (fn + tp)

In [ ]:
def calculate_ece(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0
    n = len(y_true)
    for i, (bin_lower, bin_upper) in enumerate(zip(bins[:-1], bins[1:])):
        if i == n_bins - 1:
            mask = (y_prob >= bin_lower) & (y_prob <= bin_upper)
        else:
            mask = (y_prob >= bin_lower) & (y_prob < bin_upper)
        if np.sum(mask) > 0:
            bin_conf = np.mean(y_prob[mask])
            bin_acc = np.mean(y_true[mask])
            ece += np.abs(bin_conf - bin_acc) * np.sum(mask)
    return ece / n

def calculate_mce(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    max_error = 0
    for i, (bin_lower, bin_upper) in enumerate(zip(bins[:-1], bins[1:])):
        if i == n_bins - 1:
            mask = (y_prob >= bin_lower) & (y_prob <= bin_upper)
        else:
            mask = (y_prob >= bin_lower) & (y_prob < bin_upper)
        if np.sum(mask) > 0:
            bin_conf = np.mean(y_prob[mask])
            bin_acc = np.mean(y_true[mask])
            max_error = max(max_error, np.abs(bin_conf - bin_acc)) 
    return max_error

In [ ]:
def sigmoid_numpy(x):
    return 1 / (1 + np.exp(-x))

def show_calibration_curve(y_true, y_proba):
    prob_true_svm, prob_pred_svm = calibration_curve(y_true, y_proba)

    CalibrationDisplay(prob_true_svm, prob_pred_svm, y_true).plot(label='SVM', color='green')
    plt.title("Диаграмма калибровки")
    plt.xlabel("Средняя предсказанная вероятность")
    plt.ylabel("Фактическая доля положительных исходов")
    plt.legend()
    plt.grid(True)
    plt.show()

def check_calibration(y_true, y_proba):
    sigmoid_y_proba = sigmoid_numpy(y_proba)
    
    show_calibration_curve(y_true, sigmoid_y_proba)

    brier_score = brier_score_loss(y_true, sigmoid_y_proba)
    print(f"\nОценка Бриера: {brier_score:.4f}")

    ece_score = calculate_ece(y_true, sigmoid_y_proba)
    print(f"\nОценка ECE: {ece_score:.4f}")

    mce_score = calculate_mce(y_true, sigmoid_y_proba)
    print(f"\nОценка MCE: {mce_score:.4f}")

    return brier_score, ece_score, mce_score

In [209]:
results = []

In [ ]:
def evaluate_model(pipeline, X, y, groups, is_sampling=False, name="Model"):
    scores = []
    for train_idx, val_idx in gtss.split(X, y, groups=groups):
        # if is_sampling:
            
        # else:
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        print(len(train_idx), len(val_idx))
        
        pipeline.fit(X_train, y_train)
        
        y_pred = pipeline.predict(X_val)
        y_proba = pipeline.predict_proba(X_val)[:, 1]

        roc_auc = roc_auc_score(y_val, y_proba)
    
        tn, _, fn, tp = confusion_matrix(y_val, y_pred).ravel()
        approval_rate = get_approval_rate(fn, tn, len(y_val))
        default_rate = get_default_rate(fn, tn)
        missed_defaults_rate = get_missed_defaults_rate(fn, tp)

        metrics = {
            "roc_auc": roc_auc,
            "approval_rate": approval_rate,
            "default_rate": default_rate,
            "missed_defaults_rate": missed_defaults_rate
        }

        display(pd.DataFrame([metrics]))
        scores.append(metrics)

    scores_df = pd.DataFrame(scores)
    results.append({
        'Название модели': name,
        "roc_auc": scores_df["roc_auc"].mean(),
        "approval_rate": scores_df["approval_rate"].mean(),
        "default_rate": scores_df["default_rate"].mean(),
        "missed_defaults_rate": scores_df["missed_defaults_rate"].mean()
    })

In [222]:
pipeline = Pipeline(steps=[
    ('preprocessor', linear_preprocessor),
    ('model', LogisticRegression(random_state=RANDOM_STATE))
])

evaluate_model(pipeline, X, y, groups, name="Model")

In [214]:
def run_optuna(build_pipeline_fn, n_trials=5):
    def objective(trial):
        pipeline = build_pipeline_fn(trial)

        scores = []
        for train_idx, val_idx in gtss.split(X, y, groups=groups):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            
            pipeline.fit(X_train, y_train)
            
            y_pred = pipeline.predict(X_val)

            _, _, fn, tp = confusion_matrix(y_val, y_pred).ravel()
            missed_defaults_rate = get_missed_defaults_rate(fn, tp)
            
            scores.append(missed_defaults_rate)

        mean_score = scores.mean()

        trial.report(mean_score, step=0)
        if trial.should_prune():
            raise optuna.TrialPruned()

        return mean_score

    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    study = optuna.create_study(direction='minimize', sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True, n_jobs=-1)

    fig1 = optuna.visualization.plot_optimization_history(study)
    fig2 = optuna.visualization.plot_param_importances(study)

    display(fig1)
    display(fig2)

    best_params = study.best_params
    print("Лучшие гиперпараметры:", best_params)
    best_value = study.best_value
    print("Лучшее среднее значение Missed Defaults Rate на кросс-валидации:", round(best_value, 3))

    # evaluate_model(build_best_pipeline_fn(best_params), X_train, y_train, name=name)

    return best_params

## Калибровка модели и пересчёт результатов

* Проведите калибровку лучшей версии модели. Используйте отдельную калибровочную выборку.
* Используйте метод, подходящий для случайного леса.
* Постройте график калибровки.
* Сделайте вывод, оцените результаты с помощью коэффициента Бриера.

## Поиск порога решения

* Используя откалиброванную модель и калибровочную выборку, найдите порог, при котором будут достигнуты заданные в постановке задачи значения метрик:
    * approval rate - не менее 65%;
    * default rate - не более 2%;
    * missed defaults rate - не более 4%.
    
* Сделайте вывод о достигнутых в этом разделе результатах.

## Анализ матрицы ошибок

* Оцените стабильность модели на тестовых данных. Для этого постройте:
    * матрицу ошибок на калибровочных данных;
    * матрицу классификации на тестовых данных.
* Сделайте вывод о моделях, рассчитав классические метрики машинного обучения и указанные в ТЗ бизнес-метрики.
* Сделайте вывод о стабильности модели.

## Фиксирование итоговой модели

- Опишите лучшую модель и найденный порог классификации.


## Анализ важности признаков

* Проведите анализ важности признаков найденной модели на полных тренировочных данных.
* Используйте `feature_importances_` для найденной модели.
* Сделайте вывод о силе влияния признаков на дефолт.

## Выводы по проекту

Сделайте выводы по проекту. Можете использовать такой план:

1. Цель и задачи исследования.

2. Подготовка данных и выборок.

3. Поиск и настройка модели.

4. Калибровка вероятностей.

5. Оптимизация бизнес-порога.

6. Анализ важности признаков.

7. Финальный пайплайн.

8. Основные выводы и рекомендации для бизнеса.

